In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/s5e10-tabm-over-residuals/oof_tabm_overresid.csv
/kaggle/input/s5e10-tabm-over-residuals/__results__.html
/kaggle/input/s5e10-tabm-over-residuals/__notebook__.ipynb
/kaggle/input/s5e10-tabm-over-residuals/__output__.json
/kaggle/input/s5e10-tabm-over-residuals/test_tabm_overresid.csv
/kaggle/input/s5e10-tabm-over-residuals/custom.css
/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv
/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv
/kaggle/input/pss5e10-main/cat_20seed_test_residuals.csv
/kaggle/input/pss5e10-main/__results__.html
/kaggle/input/pss5e10-main/lgb_20seed_test_residuals.csv
/kaggle/input/pss5e10-main/xgb_20seed_test_residuals.csv
/kaggle/input/pss5e10-main/__notebook__.ipynb
/kaggle/input/pss5e10-main/__output__.json
/kaggle/input/pss5e10-main/xgb_20seed_oof_residuals.csv
/kaggle/input/pss5e10-main/Train_org.csv
/kaggle/input/pss5e10-main/test.csv
/kaggle/input/pss5e10-main/final_train.csv
/kaggle/input/pss5e10-main/custom.css
/kaggle/input/pss5e1

In [2]:
!pip install autogluon.tabular[0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.3/487.3 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 7.1.0
    Uninstalling psutil-7.1.0:
      Successfully uninstalled psutil-7.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.

In [3]:
# train = pd.read_csv('/kaggle/input/pss5e10-main/final_train.csv')
# # train = train.fillna(0)
# test = pd.read_csv('/kaggle/input/pss5e10-main/test.csv')
# test = test.drop(columns='accident_risk')
# train['residual_risk'] = train['accident_risk'] - train['y']
# train.drop(columns='accident_risk', inplace=True)

In [4]:
oofs_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_baseline.csv')
test_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_baseline.csv')
oofs_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_residuals.csv')
test_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_residuals.csv')

oofs_df_residuals.columns = [col + '_res' for col in oofs_df_residuals.columns]
test_df_residuals.columns = [col + '_res' for col in test_df_residuals.columns]

oofs_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/oof_tabm_plus_origcol_tuned.csv')
test_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/test_tabm_plus_origcol_tuned.csv')

oofs_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv')
test_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv')

oofs_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv')
test_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv')

oofs_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/oof_realmlp_plus_origcol.csv')
test_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/test_realmlp_plus_origcol.csv')

oofs_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/oof_tabm_overresid.csv')
test_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/test_tabm_overresid.csv')

oofs_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_oof_residuals.csv')
test_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_test_residuals.csv')

oofs_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv')
test_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_test_residuals.csv')

oofs_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv')
test_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_test_residuals.csv')

oofs_df = pd.concat([
    oofs_df_tabm.drop(columns='id').add_prefix('tabm_'),
    oofs_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    oofs_df_xgb.drop(columns='id').add_prefix('xgb_'),
    oofs_df_mlp.drop(columns='id').add_prefix('mlp_'),
    oofs_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # oofs_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    oofs_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    oofs_df_lgb2.drop(columns='id').add_prefix('lgb2_')
], axis=1)

test_df = pd.concat([
    test_df_tabm.drop(columns='id').add_prefix('tabm_'),
    test_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    test_df_xgb.drop(columns='id').add_prefix('xgb_'),
    test_df_mlp.drop(columns='id').add_prefix('mlp_'),
    test_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # test_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    test_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    test_df_lgb2.drop(columns='id').add_prefix('lgb2_')
], axis=1)

# oofs_df = pd.concat([oofs_df_baseline, oofs_df_residuals], axis=1)
# test_df = pd.concat([test_df_baseline, test_df_residuals], axis=1)

train = pd.read_csv('/kaggle/input/playground-series-s5e10/train.csv')
y = train['accident_risk']

In [5]:
TARGET = 'accident_risk'
FEATURES = [col for col in oofs_df.columns if col!='accident_risk']

In [6]:
oofs_df = pd.concat([oofs_df, y], axis=1)

In [7]:
# train = train.fillna(0)
# test = test.fillna(0)

In [8]:
import shutil, os
old = '/kaggle/working/AutogluonModels'
if os.path.exists(old):
    shutil.rmtree(old)

In [9]:
import autogluon.core.utils.utils as core_utils
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, LeaveOneGroupOut

_ORIG_CVSPLITTER_INIT = core_utils.CVSplitter.__init__

def _cvsplitter_init_with_42(self, splitter_cls=None, n_splits=5, n_repeats=1,
                             random_state=None, stratify=False, bin=False,
                             n_bins=None, groups=None):
    # force our seed, ignore the 0 that the trainer passes
    _ORIG_CVSPLITTER_INIT(
        self,
        splitter_cls=splitter_cls,
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42,        # <-- your seed
        stratify=stratify,
        bin=bin,
        n_bins=n_bins,
        groups=groups,
    )

core_utils.CVSplitter.__init__ = _cvsplitter_init_with_42

In [10]:
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

# PEAK_XGB = {
#     'n_estimators': 100_000, 'learning_rate': 0.01, 'max_depth': 6,
#     'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
#     'tree_method': 'gpu_hist', 'device': 'cuda', 'n_jobs': -1, 'verbosity': 0
# }

# PEAK_LGB = {
#     'n_estimators': 100_000, 'learning_rate': 0.01, 'num_leaves': 64,
#     'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
#     'device': 'gpu', 'n_jobs': -1, 'verbosity': -1
# }

# PEAK_CAT = {
#     'iterations': 100_000, 'learning_rate': 0.01, 'depth': 6,
#     'l2_leaf_reg': 0.0, 'subsample': 0.9, 'task_type': 'GPU', 'verbose': False
# }

# ----------  AutoGluon search space  ----------
predictor = TabularPredictor(
    label=TARGET,
    eval_metric='rmse',
    problem_type='regression',
    path='AutogluonModels/full_hpo'
).fit(
    train_data=oofs_df,
    time_limit=39600,  # 2 hours for extensive HPO
    presets='best_quality',
    num_bag_folds=5,
    num_stack_levels=3,
    num_bag_sets=3,
    auto_stack=True,
    raise_on_no_models_fitted=False,
    # REMOVED: hyperparameter_tune=True,  # Not needed - just use hyperparameter_tune_kwargs
    # hyperparameter_tune_kwargs={
    #     'scheduler': 'local',
    #     'searcher': 'bayesopt',
    #     'num_trials': 50,
    # },
    # hyperparameters={
    #     # XGBoost with search space
    #     'XGB': {
    #         'n_estimators': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'max_depth': [4, 5, 6, 7, 8, 9],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    #         'reg_alpha': [0, 0.1, 0.5, 1.0],
    #         'reg_lambda': [0, 0.1, 0.5, 1.0, 5.0],
    #         'min_child_weight': [1, 3, 5, 7],
    #         'tree_method': 'gpu_hist',
    #         'device': 'cuda',
    #     },
        
    #     # LightGBM with search space
    #     'GBM': {
    #         'n_estimators': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'num_leaves': [31, 63, 127, 255],
    #         'max_depth': [6, 8, 10, 12, -1],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    #         'reg_alpha': [0, 0.1, 0.5, 1.0],
    #         'reg_lambda': [0, 0.1, 0.5, 1.0, 5.0],
    #         'min_child_samples': [5, 10, 20, 30],
    #         'device': 'gpu',
    #     },
        
    #     # CatBoost with search space
    #     'CAT': {
    #         'iterations': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'depth': [4, 5, 6, 7, 8, 9],
    #         'l2_leaf_reg': [1, 3, 5, 7, 9],
    #         'random_strength': [0.1, 0.5, 1.0, 2.0],
    #         'bagging_temperature': [0, 0.5, 1.0],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'task_type': 'GPU',
    #     },
        
    #     # Neural networks with tuning
    #     'NN_TORCH': {
    #         'num_layers': [2, 3, 4],
    #         'hidden_size': [128, 256, 512],
    #         'dropout_prob': [0.0, 0.1, 0.2, 0.3],
    #         'learning_rate': [1e-4, 1e-3, 1e-2],
    #         'num_epochs': [50, 100, 150],
    #         'activation': ['relu', 'elu', 'tanh', 'leaky_relu'],
    #         'use_batchnorm': [True, False],
    #     },
        
        # # Random Forest with tuning
        # 'RF': {
        #     'n_estimators': [100, 200, 300, 500],
        #     'max_depth': [10, 15, 20, None],
        #     'max_features': [0.5, 0.7, 0.9, 'auto', 'sqrt'],
        #     'min_samples_split': [2, 5, 10],
        #     'min_samples_leaf': [1, 2, 4],
        #     'bootstrap': [True, False],
        # },
        
        # # Extra Trees with tuning
        # 'XT': {
        #     'n_estimators': [100, 200, 300, 500],
        #     'max_depth': [10, 15, 20, None],
        #     'max_features': [0.5, 0.7, 0.9, 'auto', 'sqrt'],
        #     'min_samples_split': [2, 5, 10],
        #     'min_samples_leaf': [1, 2, 4],
        #     'bootstrap': [True, False],
        # },
        
        # # KNN with tuning
        # 'KNN': {
        #     'n_neighbors': [3, 5, 7, 10, 15, 20, 30, 50],
        #     'weights': ['uniform', 'distance'],
        #     'metric': ['euclidean', 'minkowski', 'manhattan'],
        # },
        
        # # Linear models with tuning
        # 'LR': {
        #     'fit_intercept': [True, False],
        #     'normalize': [True, False],
        #     'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
        # }
    # },
    verbosity=2,
    num_gpus=1
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.75 GB / 31.35 GB (94.9%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=3, num_bag_folds=5, num_bag_sets=3
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the da

[1000]	valid_set's rmse: 0.0556064
[2000]	valid_set's rmse: 0.0555976
[3000]	valid_set's rmse: 0.0555959
[4000]	valid_set's rmse: 0.0555961
[1000]	valid_set's rmse: 0.0561695
[2000]	valid_set's rmse: 0.0561573
[3000]	valid_set's rmse: 0.056157
[1000]	valid_set's rmse: 0.0562841
[2000]	valid_set's rmse: 0.0562684
[3000]	valid_set's rmse: 0.0562621
[4000]	valid_set's rmse: 0.0562617
[5000]	valid_set's rmse: 0.0562594
[6000]	valid_set's rmse: 0.0562598
[1000]	valid_set's rmse: 0.055895
[2000]	valid_set's rmse: 0.0558838
[3000]	valid_set's rmse: 0.0558811
[4000]	valid_set's rmse: 0.0558801
[5000]	valid_set's rmse: 0.0558811
[1000]	valid_set's rmse: 0.0559856
[2000]	valid_set's rmse: 0.0559743
[3000]	valid_set's rmse: 0.0559716
[4000]	valid_set's rmse: 0.0559726
[1000]	valid_set's rmse: 0.0561801
[2000]	valid_set's rmse: 0.0561696
[3000]	valid_set's rmse: 0.0561667
[4000]	valid_set's rmse: 0.0561661
[5000]	valid_set's rmse: 0.0561652
[6000]	valid_set's rmse: 0.0561651
[7000]	valid_set's rms

	-0.056	 = Validation score   (-root_mean_squared_error)
	768.25s	 = Training   runtime
	251.48s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 2270.81s of the 8867.78s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	46.15s	 = Training   runtime
	5.97s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 2218.00s of the 8814.97s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0564	 = Validation score   (-root_mean_squared_error)
	890.41s	 = Training   runtime
	20.47s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 1305.55s of the 7902.52s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will 

[1000]	valid_set's rmse: 0.0555937
[1000]	valid_set's rmse: 0.0561352
[2000]	valid_set's rmse: 0.056132
[1000]	valid_set's rmse: 0.0562739
[2000]	valid_set's rmse: 0.0562691
[3000]	valid_set's rmse: 0.056268
[4000]	valid_set's rmse: 0.0562654
[5000]	valid_set's rmse: 0.0562631
[6000]	valid_set's rmse: 0.0562616
[7000]	valid_set's rmse: 0.0562627
[1000]	valid_set's rmse: 0.055869
[1000]	valid_set's rmse: 0.0559764
[2000]	valid_set's rmse: 0.0559734
[3000]	valid_set's rmse: 0.0559723
[1000]	valid_set's rmse: 0.0561493
[1000]	valid_set's rmse: 0.0556947
[1000]	valid_set's rmse: 0.0558602
[2000]	valid_set's rmse: 0.0558509
[3000]	valid_set's rmse: 0.0558515
[1000]	valid_set's rmse: 0.056029
[2000]	valid_set's rmse: 0.0560242
[3000]	valid_set's rmse: 0.0560205
[4000]	valid_set's rmse: 0.0560189
[5000]	valid_set's rmse: 0.0560162
[6000]	valid_set's rmse: 0.056016
[7000]	valid_set's rmse: 0.0560158
[8000]	valid_set's rmse: 0.0560171
[1000]	valid_set's rmse: 0.056312
[1000]	valid_set's rmse: 0

	-0.056	 = Validation score   (-root_mean_squared_error)
	666.29s	 = Training   runtime
	189.26s	 = Validation runtime
Fitting model: LightGBM_BAG_L2 ... Training model for up to 2073.12s of the 5738.26s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	79.9s	 = Training   runtime
	8.91s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L2 ... Training model for up to 1983.39s of the 5648.53s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0559	 = Validation score   (-root_mean_squared_error)
	1883.73s	 = Training   runtime
	22.33s	 = Validation runtime
Fitting model: CatBoost_BAG_L2 ... Training model for up to 75.11s of the 3740.25s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will us

...

	-0.0559	 = Validation score   (-root_mean_squared_error)
	612.33s	 = Training   runtime
	198.45s	 = Validation runtime
Fitting model: LightGBM_BAG_L4 ... Training model for up to 408.57s of the 408.54s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0558	 = Validation score   (-root_mean_squared_error)
	55.93s	 = Training   runtime
	6.27s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L4 ... Training model for up to 345.65s of the 345.61s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0564	 = Validation score   (-root_mean_squared_error)
	337.95s	 = Training   runtime
	5.67s	 = Validation runtime
Fitting model: WeightedEnsemble_L5 ... Training model for up to 360.00s of the 1.37s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 

[1000]	valid_set's rmse: 0.0561282
[2000]	valid_set's rmse: 0.0561125
[3000]	valid_set's rmse: 0.0561088
[4000]	valid_set's rmse: 0.0561094
[1000]	valid_set's rmse: 0.055965
[2000]	valid_set's rmse: 0.055951
[3000]	valid_set's rmse: 0.0559491
[1000]	valid_set's rmse: 0.0560829
[2000]	valid_set's rmse: 0.0560751
[3000]	valid_set's rmse: 0.0560757
[1000]	valid_set's rmse: 0.0558858
[2000]	valid_set's rmse: 0.0558804
[3000]	valid_set's rmse: 0.0558792
[1000]	valid_set's rmse: 0.0558321
[2000]	valid_set's rmse: 0.055815
[3000]	valid_set's rmse: 0.0558096
[4000]	valid_set's rmse: 0.0558069
[5000]	valid_set's rmse: 0.0558082
[1000]	valid_set's rmse: 0.0562832
[2000]	valid_set's rmse: 0.0562748
[3000]	valid_set's rmse: 0.0562727
[4000]	valid_set's rmse: 0.0562706
[5000]	valid_set's rmse: 0.056271
[1000]	valid_set's rmse: 0.0559559
[2000]	valid_set's rmse: 0.0559407
[3000]	valid_set's rmse: 0.0559361
[4000]	valid_set's rmse: 0.0559334
[5000]	valid_set's rmse: 0.0559336
[1000]	valid_set's rmse:

	-0.0559	 = Validation score   (-root_mean_squared_error)
	830.71s	 = Training   runtime
	297.7s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 27818.35s of the 27818.34s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	53.49s	 = Training   runtime
	7.13s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 27757.03s of the 27757.02s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0563	 = Validation score   (-root_mean_squared_error)
	1046.43s	 = Training   runtime
	22.69s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 26684.15s of the 26684.14s of remaining time.
Specified total num_gpus: 1, but only 0 are available

[1000]	valid_set's rmse: 0.055923


	-0.0559	 = Validation score   (-root_mean_squared_error)
	185.51s	 = Training   runtime
	44.48s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 10494.24s of the 10494.23s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	Ran out of time, stopping training early. (Stopping on epoch 19)
	Ran out of time, stopping training early. (Stopping on epoch 19)
	Ran out of time, stopping training early. (Stopping on epoch 20)
	Ran out of time, stopping training early. (Stopping on epoch 20)
	Ran out of time, stopping training early. (Stopping on epoch 20)
	Ran out of time, stopping training early. (Stopping on epoch 20)
	Ran out of time, stopping training early. (Stopping on epoch 21)
	Ran out of time, stopping training early. (Stopping on epoch 22)
	Ran out of time, stopping training early. 

In [11]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.055874,root_mean_squared_error,37.852570,11737.854278,0.009480,0.916814,2,True,16
1,NeuralNetFastAI_BAG_L1,-0.055886,root_mean_squared_error,9.260061,3139.142528,9.260061,3139.142528,1,True,6
2,LightGBM_BAG_L1,-0.055903,root_mean_squared_error,7.128428,53.492722,7.128428,53.492722,1,True,2
3,CatBoost_r9_BAG_L1,-0.055904,root_mean_squared_error,6.725637,516.594779,6.725637,516.594779,1,True,14
4,NeuralNetTorch_r79_BAG_L1,-0.055905,root_mean_squared_error,4.853865,8472.319328,4.853865,8472.319328,1,True,11
5,LightGBM_r131_BAG_L1,-0.055908,root_mean_squared_error,44.483963,185.513717,44.483963,185.513717,1,True,12
6,NeuralNetFastAI_r191_BAG_L1,-0.055911,root_mean_squared_error,27.083591,9760.422933,27.083591,9760.422933,1,True,13
7,CatBoost_BAG_L1,-0.055914,root_mean_squared_error,0.583475,349.105879,0.583475,349.105879,1,True,4
8,CatBoost_r177_BAG_L1,-0.055919,root_mean_squared_error,0.481209,275.075964,0.481209,275.075964,1,True,10
9,XGBoost_BAG_L1,-0.055929,root_mean_squared_error,2.835541,42.701534,2.835541,42.701534,1,True,7


In [12]:
models = leaderboard['model'].to_list()
oofs_dict = {}
for m in models:
    oofs_dict[m] = predictor.predict_oof(model=m)

oofs_df = pd.DataFrame(oofs_dict)

In [13]:
predictor2 = TabularPredictor.load('/kaggle/working/AutogluonModels/full_hpo')
test_preds = {}
for m in models:
    test_preds[m] = predictor2.predict(test_df, model=m)

test_preds_df = pd.DataFrame(test_preds)

In [14]:
# for col in oofs_df.columns:
#     oofs_df[col] = oofs_df[col] + train['y']
#     test_preds_df[col] = test_preds_df[col] + test['y']

In [15]:
samp = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
samp['accident_risk'] = test_preds_df.iloc[:,0]
samp.to_csv('autogluon_meta_39_models.csv', index=False)

In [16]:
leaderboard.to_csv('leaderboard_autogluon_residuals.csv', index=False)
oofs_df.to_csv('oofs_autogluon_residuals.csv', index=False)
test_preds_df.to_csv('test_preds_autogluon_residuals.csv', index=False)